In [2]:
def positive_residue_percentage(sequence: str) -> float:
    """
    Calculate the percentage of positively charged residues (K, R, H)
    in a given protein sequence.

    Parameters:
        sequence (str): Protein sequence using single-letter amino acid codes.

    Returns:
        float: Percentage of positively charged residues.
    """
    # Ensure sequence is uppercase
    sequence = sequence.upper().strip()

    # Define positively charged amino acids
    positive_residues = {'K', 'R', 'H'}

    # Count total amino acids and positive ones
    total = len(sequence)
    if total == 0:
        raise ValueError("Protein sequence is empty.")

    positive_count = sum(aa in positive_residues for aa in sequence)

    # Calculate percentage
    percentage = (positive_count / total) * 100
    return percentage

sequences = {'Ect2 PBC':"LSRASRAIKKTSKKVTRAFSFTKTPKRALQR"}

for name, seq in sequences.items():
    print(f'Percentage of positively charged residues in {name}: {positive_residue_percentage(seq):.2f}%')

Percentage of positively charged residues in Ect2 PBC: 35.48%


In [20]:
def find_positive_regions(sequence: str, min_len: int = 25, max_len: int = 45, threshold: float = 20.0):
    """
    Identify and merge positively charged regions in a protein sequence.

    Parameters:
        sequence (str): Protein sequence.
        min_len (int): Minimum window size to test.
        max_len (int): Maximum window size to test.
        threshold (float): Minimum % positive residues to report.

    Returns:
        list of dicts: Each dict has 'start', 'end', 'length', '%positive', and 'sequence'.
    """
    sequence = sequence.upper().strip()
    positive_residues = {'K', 'R', 'H'}
    seq_len = len(sequence)
    raw_hits = []

    # Step 1: Identify all windows above threshold
    for window_size in range(min_len, max_len + 1):
        for start in range(seq_len - window_size + 1):
            end = start + window_size
            window_seq = sequence[start:end]
            positive_count = sum(aa in positive_residues for aa in window_seq)
            percent_positive = (positive_count / window_size) * 100

            if percent_positive >= threshold:
                raw_hits.append((start, end))

    if not raw_hits:
        return []

    # Step 2: Sort and merge overlapping or adjacent hits
    raw_hits.sort(key=lambda x: x[0])
    merged = [raw_hits[0]]

    for current in raw_hits[1:]:
        prev_start, prev_end = merged[-1]
        curr_start, curr_end = current

        if curr_start <= prev_end:  # overlap or adjacency
            merged[-1] = (prev_start, max(prev_end, curr_end))
        else:
            merged.append(current)

    # Step 3: Summarize merged regions
    merged_regions = []
    for start, end in merged:
        region_seq = sequence[start:end]
        length = end - start
        positive_count = sum(aa in positive_residues for aa in region_seq)
        percent_positive = (positive_count / length) * 100

        merged_regions.append({
            'start': start + 1,  # 1-indexed
            'end': end,
            'length': length,
            'percent_positive': round(percent_positive, 2),
            'sequence': region_seq
        })

    return merged_regions



sequences = {'abr': "MEPVSHQDMPRLSWIDTLYSNFNYGTDGYDAEGNEEHKNSREGSETMPYIDESPTMSPQLSARSQDSVDGVSPTPTEVLLPGGESESDKGLLMRKLVLSGVLASEEIYINQLEALLLPMKPLKATASTSQPVLTLQQINDIFYKIEDIYQMHKDFYDKLCPIVQQWDNKTTVGHLFQKLATQLGVYKAFVDNYKFALETAEKCSQCNVQFFKISEDLKVKGPKDSKEQPQSVTMEALLYKPIDRVTRSTLVLHDLLKHTPTDHPDYPLLQDALRISQNFLSSINEDIDPRRTAVTTPKGEPRQLVKDGFLVELSENSRKLRHLFLFTDLLLCAKLKKTTVGKHQQYDCKWYIPLADLVFPSLEESEPIHQLHATPDYEIEEMKAKISVLKSEIQKEKKSNKGSSRAIERLKKKMFEYESWLLLYSPTIPFRIHNKNGKSYLFLLSSDYERSEWREAIQKLQKKDLQALALSPFELQVLTASCFKLRTVHNVPIISHKDDDESPGLYGFLHVIVKSAKGFSHSSNFYCTLEVDSFGYFVSKAKTRVFRDTSEPEWNEEFEIELEGSQCLRILCYETCYDKSKLNKDNNEIVDKIMGKGQIQLDPQGVQSKNWHDDVIEMNGIKVEFSMKFSSRDMSLKRTPSKKQTGVFGVKISVVTKRERSKVPYIVRQCIEEVEKRGIEEVGIYRISGVATDIQALKAAFDANSKDILMMLSDMDINAIAGTLKLYFRELPEPLLTDRLYLAFMEGIALSDPAAKENCMMHLLRSLPDPNLITFLFLLHHLKKVAENEPINKMSLHNLATVFGPTLLRPSEVEIKGHMNLASDIWSHDVMAQVQVLLYYLQHPPISFSELKRSTLYYSTDV",
             'AKAP13': "MKLNPQQAPLYGDCVVTVLLAEEDKAEDDVVFYLVFLGSTLRHCTSTRKVSSDTLETIAPGHDCCETVKVQLCASKEGLPVFVVAEEDFHFVQDEAYDAAQFLATSAGNQQALNFTRFLDQSGPPSGDVNSLDKKLVLAFRHLKLPTEWNVLGTDQSLHDAGPRETLMHFAVRLGLLRLTWFLLQKPGGRGALSIHNQEGATPVSLALERGYHKLHQLLTEENAGEPDSWSSLSYEIPYGDCSVRHHRELDIYTLTSESDSHHEHPFPGDGCTGPIFKLMNIQQQLMKTNLKQMDSLMPLMMTAQDPSSAPETDGQFLPCAPEPTDPQRLSSSEETESTQCCPGSPVAQTESPCDLSSIVEEENTDRSCRKKNKGVERKGEEVEPAPIVDSGTVSDQDSCLQSLPDCGVKGTEGLSSCGNRNEETGTKSSGMPTDQESLSSGDAVLQRDLVMEPGTAQYSSGGELGGISTTNVSTPDTAGEMEHGLMNPDATVWKNVLQGGESTKERFENSNIGTAGASDVHVTSKPVDKISVPNCAPAASSLDGNKPAESSLAFSNEETSTEKTAETETSRSREESADAPVDQNSVVIPAAAKDKISDGLEPYTLLAAGIGEAMSPSDLALLGLEEDVMPHQNSETNSSHAQSQKGKSSPICSTTGDDKLCADSACQQNTVTSSGDLVAKLCDNIVSESESTTARQPSSQDPPDASHCEDPQAHTVTSDPVRDTQERADFCPFKVVDNKGQRKDVKLDKPLTNMLEVVSHPHPVVPKMEKELVPDQAVISDSTFSLANSPGSESVTKDDALSFVPSQKEKGTATPELHTATDYRDGPDGNSNEPDTRPLEDRAVGLSTSSTAAELQHGMGNTSLTGLGGEHEGPAPPAIPEALNIKGNTDSSLQSVGKATLALDSVLTEEGKLLVVSESSAAQEQDKDKAVTCSSIKENALSSGTLQEEQRTPPPGQDTQQFHEKSISADCAKDKALQLSNSPGASSAFLKAETEHNKEVAPQVSLLTQGGAAQSLVPPGASLATESRQEALGAEHNSSALLPCLLPDGSDGSDALNCSQPSPLDVGVKNTQSQGKTSACEVSGDVTVDVTGVNALQGMAEPRRENISHNTQDILIPNVLLSQEKNAVLGLPVALQDKAVTDPQGVGTPEMIPLDWEKGKLEGADHSCTMGDAEEAQIDDEAHPVLLQPVAKELPTDMELSAHDDGAPAGVREVMRAPPSGRERSTPSLPCMVSAQDAPLPKGADLIEEAASRIVDAVIEQVKAAGALLTEGEACHMSLSSPELGPLTKGLESAFTEKVSTFPPGESLPMGSTPEEATGSLAGCFAGREEPEKIILPVQGPEPAAEMPDVKAEDEVDFRASSISEEVAVGSIAATLKMKQGPMTQAINRENWCTIEPCPDAASLLASKQSPECENFLDVGLGRECTSKQGVLKRESGSDSDLFHSPSDDMDSIIFPKPEEEHLACDITGSSSSTDDTASLDRHSSHGSDVSLSQILKPNRSRDRQSLDGFYSHGMGAEGRESESEPADPGDVEEEEMDSITEVPANCSVLRSSMRSLSPFRRHSWGPGKNAASDAEMNHRSSMRVLGDVVRRPPIHRRSFSLEGLTGGAGVGNKPSSSLEVSSANAEELRHPFSGEERVDSLVSLSEEDLESDQREHRMFDQQICHRSKQQGFNYCTSAISSPLTKSISLMTISHPGLDNSRPFHSTFHNTSANLTESITEENYNFLPHSPSKKDSEWKSGTKVSRTFSYIKNKMSSSKKSKEKEKEKDKIKEKEKDSKDKEKDKKTVNGHTFSSIPVVGPISCSQCMKPFTNKDAYTCANCSAFVHKGCRESLASCAKVKMKQPKGSLQAHDTSSLPTVIMRNKPSQPKERPRSAVLLVDETATTPIFANRRSQQSVSLSKSVSIQNITGVGNDENMSNTWKFLSHSTDSLNKISKVNESTESLTDEGVGTDMNEGQLLGDFEIESKQLEAESWSRIIDSKFLKQQKKDVVKRQEVIYELMQTEFHHVRTLKIMSGVYSQGMMADLLFEQQMVEKLFPCLDELISIHSQFFQRILERKKESLVDKSEKNFLIKRIGDVLVNQFSGENAERLKKTYGKFCGQHNQSVNYFKDLYAKDKRFQAFVKKKMSSSVVRRLGIPECILLVTQRITKYPVLFQRILQCTKDNEVEQEDLAQSLSLVKDVIGAVDSKVASYEKKVRLNEIYTKTDSKSIMRMKSGQMFAKEDLKRKKLVRDGSVFLKNAAGRLKEVQAVLLTDILVFLQEKDQKYIFASLDQKSTVISLKKLIVREVAHEEKGLFLISMGMTDPEMVEVHASSKEERNSWIQIIQDTINTLNRDEDEGIPSENEEEKKMLDTRARELKEQLHQKDQKILLLLEEKEMIFRDMAECSTPLPEDCSPTHSPRVLFRSNTEEALKGGPLMKSAINEVEILQGLVSGNLGGTLGPTVSSPIEQDVVGPVSLPRRAETFGGFDSHQMNASKGGEKEEGDDGQDLRRTESDSGLKKGGNANLVFMLKRNSEQVVQSVVHLYELLSALQGVVLQQDSYIEDQKLVLSERALTRSLSRPSSLIEQEKQRSLEKQRQDLANLQKQQAQYLEEKRRREREWEARERELREREALLAQREEEVQQGQQDLEKEREELQQKKGTYQYDLERLRAAQKQLEREQEQLRREAERLSQRQTERDLCQVSHPHTKLMRIPSFFPSPEEPPSPSAPSIAKSGSLDSELSVSPKRNSISRTHKDKGPFHILSSTSQTNKGPEGQSQAPASTSASTRLFGLTKPKEKKEKKKKNKTSRSQPGDGPASEVSAEGEEIFC",
             'alpha-pix': "MNPEEQIVTWLISLGVLESPKKTICDPEEFLKSSLKNGVVLCKLINRLMPGSVEKFCLDPQTEADCINNINDFLKGCATLQVEIFDPDDLYSGVNFSKVLSTLLAVNKATEDQLSERPCGRSSSLSAANTSQTNPQGAVSSTVSGLQRQSKTVEMTENGSHQLIVKARFNFKQTNEDELSVCKGDIIYVTRVEEGGWWEGTLNGRTGWFPSNYVREIKSSERPLSPKAVKGFETAPLTKNYYTVVLQNILDTEKEYAKELQSLLVTYLRPLQSNNNLSTVEVTSLLGNFEEVCTFQQTLCQALEECSKFPENQHKVGGCLLSLMPHFKSMYLAYCANHPSAVNVLTQHSDELEQFMENQGASSPGILILTTNLSKPFMRLEKYVTLLQELERHMEDTHPDHQDILKAIVAFKTLMGQCQDLRKRKQLELQILSEPIQAWEGEDIKNLGNVIFMSQVMVQYGACEEKEERYLMLFSNVLIMLSASPRMSGFIYQGKIPIAGTVVTRLDEIEGNDCTFEITGNTVERIVVHCNNNQDFQEWLEQLNRLIRGPASCSSLSKTSSSSCSAHSSFSSTGQPRGPLEPPQIIKPWSLSCLRPAPPLRPSAALGYKERMSYILKESSKSPKTMKKFLHKRKTERKPSEEEYVIRKSTAALEEDAQILKVIEAYCTSANFQQGHGSSTRKDSIPQVLLPEEEKLIIEETRSNGQTIMEEKSLVDTVYALKDEVRELKQENKRMKQCLEEELKSRRDLEKLVRRLLKQTDECIRGESSSKTSILP",
             
             }
threshold = 30.0

for name, seq in sequences.items():
    print(f"\nAnalyzing sequence: {name}")
    r = find_positive_regions(seq, threshold=threshold)
    if not r:
        print("No positively charged regions found above the threshold.\n")
        continue
    else:
        print(f"Found {len(r)} positively charged regions above {threshold}% threshold:\n")
    for region in r:
        print(f"Positions {region['start']}-{region['end']} ({region['length']} aa): {region['percent_positive']}% positive")
        print(f"  {region['sequence']}\n")
    print("__" * 40)



Analyzing sequence: abr
Found 3 positively charged regions above 30.0% threshold:

Positions 314-350 (37 aa): 27.03% positive
  SENSRKLRHLFLFTDLLLCAKLKKTTVGKHQQYDCKW

Positions 369-424 (56 aa): 25.0% positive
  HQLHATPDYEIEEMKAKISVLKSEIQKEKKSNKGSSRAIERLKKKMFEYESWLLLY

Positions 627-669 (43 aa): 27.91% positive
  MKFSSRDMSLKRTPSKKQTGVFGVKISVVTKRERSKVPYIVRQ

________________________________________________________________________________

Analyzing sequence: AKAP13
Found 11 positively charged regions above 30.0% threshold:

Positions 1560-1605 (46 aa): 26.09% positive
  PFRRHSWGPGKNAASDAEMNHRSSMRVLGDVVRRPPIHRRSFSLEG

Positions 1725-1807 (83 aa): 28.92% positive
  YNFLPHSPSKKDSEWKSGTKVSRTFSYIKNKMSSSKKSKEKEKEKDKIKEKEKDSKDKEKDKKTVNGHTFSSIPVVGPISCSQ

Positions 1828-1853 (26 aa): 30.77% positive
  HKGCRESLASCAKVKMKQPKGSLQAH

Positions 1982-2015 (34 aa): 29.41% positive
  SKFLKQQKKDVVKRQEVIYELMQTEFHHVRTLKI

Positions 2047-2080 (34 aa): 26.47% positive
  SIHSQFFQRILERKKESLVDKSEKNFLIKRIGDV

Pos